In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn gradio matplotlib pandas

In [ ]:
import re, random, json
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          EarlyStoppingCallback, pipeline)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, ConfusionMatrixDisplay, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    _, _, f1w, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision_macro": p,
            "recall_macro": r, "f1_macro": f1, "f1_weighted": f1w}

class WeightedTrainer(Trainer):
    """Trainer with optional class weights (used for imbalanced data)."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

def dedup(d, text_col="text"):
    df = d.to_pandas().drop_duplicates(subset=text_col).reset_index(drop=True)
    return Dataset.from_pandas(df, features=d.features, preserve_index=False)

def train_model(model_name, run_name, splits, id2label, lr=5e-5, epochs=5,
                batch_size=32, max_length=64, class_weights=None):
    """splits: DatasetDict with train/validation/test, columns 'text' and 'label'."""
    tok = AutoTokenizer.from_pretrained(model_name)
    tokenized = splits.map(lambda b: tok(b["text"], truncation=True, max_length=max_length), batched=True)
    tokenized = tokenized.rename_column("label", "labels")
    keep = {"input_ids", "attention_mask", "token_type_ids", "labels"}
    tokenized = tokenized.remove_columns([c for c in tokenized["train"].column_names if c not in keep])

    label2id = {v: k for k, v in id2label.items()}
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id)
    steps_per_epoch = -(-len(tokenized["train"]) // batch_size)       # ceil division
    warmup_steps = int(0.1 * steps_per_epoch * epochs)                # 10% warmup, works on all versions

    args = TrainingArguments(
        output_dir=f"runs/{run_name}", eval_strategy="epoch", save_strategy="epoch",
        learning_rate=lr, per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        num_train_epochs=epochs, weight_decay=0.01, warmup_steps=warmup_steps,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
        save_total_limit=1, fp16=torch.cuda.is_available(), report_to="none", seed=SEED)

    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
        processing_class=tok, data_collator=DataCollatorWithPadding(tok),
        compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        class_weights=class_weights)
    trainer.train()
    val = trainer.evaluate()
    test = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    return trainer, tok, tokenized, {"val": val, "test": test}

def evaluate_on(trainer, tok, dataset, max_length=64):
    """Evaluate a trained model on any dataset with 'text' and 'label' columns."""
    enc = dataset.map(lambda b: tok(b["text"], truncation=True, max_length=max_length), batched=True)
    enc = enc.rename_column("label", "labels")
    out = trainer.predict(enc)
    return compute_metrics((out.predictions, out.label_ids))

def result_row(name, m):
    return {"run": name,
            "val_f1_macro": round(m["val"]["eval_f1_macro"], 4),
            "test_accuracy": round(m["test"]["test_accuracy"], 4),
            "test_precision_macro": round(m["test"]["test_precision_macro"], 4),
            "test_recall_macro": round(m["test"]["test_recall_macro"], 4),
            "test_f1_macro": round(m["test"]["test_f1_macro"], 4),
            "test_f1_weighted": round(m["test"]["test_f1_weighted"], 4)}

def show_errors(trainer, tokenized, raw_test, id2label, n=10):
    out = trainer.predict(tokenized["test"])
    probs = torch.softmax(torch.tensor(out.predictions), dim=-1)
    y_pred, y_true = probs.argmax(-1).numpy(), out.label_ids
    conf = probs.max(-1).values.numpy()
    wrong = np.where(y_pred != y_true)[0][:n]
    for i in wrong:
        print(f"TEXT : {raw_test[int(i)]['text']}\nTRUE : {id2label[int(y_true[i])]} | "
              f"PRED : {id2label[int(y_pred[i])]} ({conf[i]:.2f})\n")
    return y_true, y_pred

In [ ]:
ds = load_dataset("PolyAI/banking77")
print(ds)
label_names = ds["train"].features["label"].names
id2label = {i: n for i, n in enumerate(label_names)}
print(len(label_names), "classes")
print(ds["train"][0])

df = ds["train"].to_pandas()
print(df["text"].str.split().str.len().describe())       # text length in words
print(df["label"].value_counts().describe())              # class balance
print("duplicates:", df["text"].duplicated().sum(), "| empty:", (df["text"].str.strip() == "").sum())

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
77 classes
{'text': 'I am still waiting on my card?', 'label': 11}
count    10003.000000
mean        11.949415
std          7.891577
min          2.000000
25%          7.000000
50%         10.000000
75%         13.000000
max         79.000000
Name: text, dtype: float64
count     77.000000
mean     129.909091
std       32.942207
min       35.000000
25%      112.000000
50%      127.000000
75%      159.000000
max      187.000000
Name: count, dtype: float64
duplicates: 0 | empty: 0


In [ ]:
def clean_en(ex):
    ex["text"] = re.sub(r"\s+", " ", ex["text"]).strip()
    return ex

train_full = dedup(ds["train"].map(clean_en))
test = dedup(ds["test"].map(clean_en))
test_texts = set(test["text"])
train_full = train_full.filter(lambda ex: ex["text"] not in test_texts)   # avoid leakage

tv = train_full.train_test_split(test_size=0.1, seed=SEED, stratify_by_column="label")
splits = DatasetDict({"train": tv["train"], "validation": tv["test"], "test": test})
print({k: len(v) for k, v in splits.items()})

Filter:   0%|          | 0/9999 [00:00<?, ? examples/s]

{'train': 8992, 'validation': 1000, 'test': 3079}


In [ ]:
# distilbert
results = []
trainer_a1, tok_a1, tok_data_a1, m_a1 = train_model(
    "distilbert-base-uncased", "A1_distilbert", splits, id2label, lr=5e-5, epochs=6)
results.append(result_row("A1 distilbert baseline", m_a1))
pd.DataFrame(results)

Map:   0%|          | 0/8992 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3079 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
1,No log,1.491245,0.756000,0.740897,0.710127,0.688292,0.727764
2,2.290000,0.530328,0.884000,0.883114,0.869653,0.868612,0.879904
3,2.290000,0.415353,0.884000,0.898708,0.880482,0.881533,0.882090
4,0.297449,0.388630,0.901000,0.912553,0.897663,0.899355,0.899393
5,0.297449,0.372409,0.905000,0.915439,0.900482,0.901312,0.902458
6,0.088090,0.383493,0.900000,0.913061,0.896867,0.897907,0.897912


... (6 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
0.088090,0.372409,6,0.905000,0.915439,0.900482,0.901312,0.902458


[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
0.088090,0.280887,6,0.926600,0.929373,0.926615,0.926357,0.926341


,run,val_f1_macro,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted
0,A1 distilbert baseline,0.9013,0.9266,0.9294,0.9266,0.9264,0.9263


In [ ]:
y_true, y_pred = show_errors(trainer_a1, tok_data_a1, splits["test"], id2label, n=10)
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

# Most confused intent pairs (a 77x77 matrix is unreadable, so list the top pairs)
pairs = Counter((id2label[int(t)], id2label[int(p)]) for t, p in zip(y_true, y_pred) if t != p)
for (t, p), c in pairs.most_common(10):
    print(f"{c:3d}  true={t}  ->  pred={p}")

TEXT : How do I locate my card?
TRUE : card_arrival | PRED : lost_or_stolen_card (0.64)

TEXT : Is there a way to know when my card will arrive?
TRUE : card_arrival | PRED : card_delivery_estimate (0.86)

TEXT : When will I get my card?
TRUE : card_arrival | PRED : card_delivery_estimate (0.98)

TEXT : How long does a card delivery take?
TRUE : card_arrival | PRED : card_delivery_estimate (0.90)

TEXT : How do I know when my card will arrive?
TRUE : card_arrival | PRED : card_delivery_estimate (0.97)

TEXT : Is there tracking info available?
TRUE : card_arrival | PRED : why_verify_identity (0.31)

TEXT : The app doesn't show the card I received.
TRUE : card_linking | PRED : card_arrival (0.86)

TEXT : Can I link another card to my account?
TRUE : card_linking | PRED : getting_spare_card (0.83)

TEXT : Why am I being charged more ?
TRUE : card_payment_wrong_exchange_rate | PRED : extra_charge_on_statement (0.34)

TEXT : the conversion value for my card payments is incorrect.
TRUE : card

**Improvement: Using a larger model**

In [ ]:
# bert_base
trainer_a2, tok_a2, tok_data_a2, m_a2 = train_model(
    "bert-base-uncased", "A2_bert_base", splits, id2label, lr=3e-5, epochs=6)
results.append(result_row("A2 bert-base-uncased", m_a2))
pd.DataFrame(results)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8992 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

... (13 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
0.286781,0.411625,6,0.905000,0.913897,0.899269,0.900998,0.903345


[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
0.286781,0.358085,6,0.924651,0.927718,0.924667,0.924727,0.924707


,run,val_f1_macro,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted
0,A1 distilbert baseline,0.9013,0.9266,0.9294,0.9266,0.9264,0.9263
1,A2 bert-base-uncased,0.9010,0.9247,0.9277,0.9247,0.9247,0.9247


**Improvement: Handling typos by testing the model on noisy text and then fixing them by augmentation**

In [ ]:
def add_typos(text, p=0.06, seed=0):
    rnd = random.Random(seed)
    chars, i = list(text), 0
    while i < len(chars) - 1:
        if chars[i].isalpha() and rnd.random() < p:
            op = rnd.choice(["swap", "drop", "dup"])
            if op == "swap": chars[i], chars[i + 1] = chars[i + 1], chars[i]
            elif op == "drop": chars.pop(i); continue
            else: chars.insert(i, chars[i]); i += 1
        i += 1
    return "".join(chars)

noisy_test = splits["test"].map(lambda ex, idx: {"text": add_typos(ex["text"], seed=idx)}, with_indices=True)
noisy_train = splits["train"].map(lambda ex, idx: {"text": add_typos(ex["text"], seed=10_000 + idx)}, with_indices=True)

before = evaluate_on(trainer_a1, tok_a1, noisy_test)          # baseline on noisy test
print("Baseline on noisy test:", before)

splits_aug = DatasetDict({"train": concatenate_datasets([splits["train"], noisy_train]),
                          "validation": splits["validation"], "test": splits["test"]})
trainer_a3, tok_a3, _, m_a3 = train_model(
    "distilbert-base-uncased", "A3_distilbert_aug", splits_aug, id2label, lr=5e-5, epochs=6)
after = evaluate_on(trainer_a3, tok_a3, noisy_test)           # augmented model on noisy test
print("Augmented on noisy test:", after)
results.append(result_row("A3 distilbert + typo augmentation (clean test)", m_a3))

robust = pd.DataFrame({"baseline_noisy_test": before, "augmented_noisy_test": after}).round(4)
robust

Map:   0%|          | 0/3079 [00:00<?, ? examples/s]

Map:   0%|          | 0/8992 [00:00<?, ? examples/s]

Map:   0%|          | 0/3079 [00:00<?, ? examples/s]

Baseline on noisy test: {'accuracy': 0.7164663851899967, 'precision_macro': 0.7519319143526891, 'recall_macro': 0.7165084915084916, 'f1_macro': 0.7215137042428099, 'f1_weighted': 0.7214732235211762}


Map:   0%|          | 0/17984 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

... (12 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro,F1 Weighted
0.016901,0.296216,6,0.929847,0.933030,0.929862,0.929935,0.929924


Map:   0%|          | 0/3079 [00:00<?, ? examples/s]

Augmented on noisy test: {'accuracy': 0.8626177330302046, 'precision_macro': 0.869809525462477, 'recall_macro': 0.8626456876456875, 'f1_macro': 0.8628345293653845, 'f1_weighted': 0.862802634454156}


,baseline_noisy_test,augmented_noisy_test
accuracy,0.7165,0.8626
precision_macro,0.7519,0.8698
recall_macro,0.7165,0.8626
f1_macro,0.7215,0.8628
f1_weighted,0.7215,0.8628


In [ ]:
# Choose the final model by validation macro-F1, then report its test numbers. Set best_trainer, best_tok to that run.
res_a = pd.DataFrame(results)
res_a.to_csv("results_project_A.csv", index=False)
res_a

,run,val_f1_macro,test_accuracy,test_precision_macro,test_recall_macro,test_f1_macro,test_f1_weighted
0,A1 distilbert baseline,0.9013,0.9266,0.9294,0.9266,0.9264,0.9263
1,A2 bert-base-uncased,0.9010,0.9247,0.9277,0.9247,0.9247,0.9247
2,A3 distilbert + typo augmentation (clean test),0.9087,0.9298,0.9330,0.9299,0.9299,0.9299


In [ ]:
# saving the model and testing on new data
best_trainer, best_tok = trainer_a3, tok_a3     # change to the run you picked
best_trainer.save_model("saved_banking_router")
best_tok.save_pretrained("saved_banking_router")

clf = pipeline("text-classification", model="saved_banking_router", top_k=3,
               device=0 if torch.cuda.is_available() else -1)
test_msgs = [
    "What is the exchange rate for euros?",
    "What is the exchange rate?",
    "How much is one euro worth?",
    "I want to exchange euros",
    "Can I exchange EUR to USD?",
    "What currencies are supported?",
    "Do you support euros?",
]

for m in test_msgs:
    res = clf(m)
    res = res[0] if isinstance(res[0], list) else res
    print(m, "->", [(r["label"], round(r["score"], 2)) for r in res])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

What is the exchange rate for euros? -> [('exchange_rate', 0.83), ('fiat_currency_support', 0.11), ('exchange_charge', 0.03)]
What is the exchange rate? -> [('exchange_rate', 1.0), ('wrong_exchange_rate_for_cash_withdrawal', 0.0), ('transfer_timing', 0.0)]
How much is one euro worth? -> [('exchange_charge', 0.41), ('extra_charge_on_statement', 0.25), ('fiat_currency_support', 0.1)]
I want to exchange euros -> [('exchange_via_app', 0.52), ('fiat_currency_support', 0.47), ('supported_cards_and_currencies', 0.0)]
Can I exchange EUR to USD? -> [('fiat_currency_support', 0.98), ('exchange_via_app', 0.02), ('supported_cards_and_currencies', 0.0)]
What currencies are supported? -> [('fiat_currency_support', 0.99), ('supported_cards_and_currencies', 0.01), ('exchange_via_app', 0.0)]
Do you support euros? -> [('fiat_currency_support', 0.98), ('country_support', 0.01), ('supported_cards_and_currencies', 0.0)]


In [ ]:
# gradio demo
import gradio as gr
def predict(text):
    res = clf(text)
    res = res[0] if isinstance(res[0], list) else res
    return {r["label"]: float(r["score"]) for r in res}

gr.Interface(fn=predict,
             inputs=gr.Textbox(lines=3, label="Customer message"),
             outputs=gr.Label(num_top_classes=3, label="Predicted intent"),
             title="Banking Support Query Router",
             examples=test_msgs).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5c8247e95a0042d0fb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!zip -r saved_banking_router.zip saved_banking_router
from google.colab import files
files.download("saved_banking_router.zip")

  adding: saved_banking_router/ (stored 0%)
  adding: saved_banking_router/config.json (deflated 71%)
  adding: saved_banking_router/tokenizer_config.json (deflated 43%)
  adding: saved_banking_router/model.safetensors (deflated 8%)
  adding: saved_banking_router/training_args.bin (deflated 53%)
  adding: saved_banking_router/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>